# 03 — Layer 2-B: PDF → FHIR (Vision Extraction)

🔧 Script (rasterization, bbox extraction) · 🤖 LLM (live extraction)

Layer 2-B handles unstructured sources. A PDF goes through six steps:

1. **Pick a PDF** — defaults to the rhett759 fixture; swap by pointing `PDF_PATH` at any file or any prior upload.
2. **Rasterize a page** — renders the chosen page to a PNG so you can see what the model sees.
3. **Extract text + bbox layer** — pdfplumber pulls every text span with its bounding box, useful as a ground-truth grid.
4. **Find a known anchor** — sanity-check that the layout tooling can locate a phrase by name (used for golden-output regression tests).
5. **Live extraction run** — actually calls the configured `VisionBackend` (Claude today; Gemma 4 in a follow-up commit) and returns a validated `ExtractionResult`.
6. **Convert to FHIR Observation** — the deterministic `to_fhir` layer turns the validated result into a profile-tagged FHIR resource with provenance extensions.

**Each cell is independent** — re-run any one without re-running earlier ones, as long as `PDF_PATH` is still defined.

**About caching:** by default this notebook runs **live every time** (`SKIP_CACHE = True`). Set it to `False` once you've reviewed an extraction and want to replay from cache.

## Step 0 — Pick a PDF

Three ways to choose:

1. **Default**: the rhett759 lab fixture ships in the corpus.
2. **Path to any PDF**: set `PDF_PATH = Path("…/your.pdf")`.
3. **Prior upload**: use `list_uploads()` and `get_upload(prefix).pdf_path`.

Optionally, drop the chosen PDF into the uploads dropbox via `store_upload_from_path()` so it's findable next session.

In [ ]:
from pathlib import Path
import json

ATLAS_ROOT = Path("..").resolve()
CORPUS = ATLAS_ROOT / "corpus"

# --- Choose your PDF here -----------------------------------------------
# Default fixture: synthesized 3-page Quest lab report (rhett759)
PDF_PATH = CORPUS / "_sources" / "synthesized-lab-pdf" / "raw" / "lab-report-2025-09-12-quest.pdf"

# Or point at any PDF on disk:
# PDF_PATH = Path("/path/to/your.pdf")

# Or pick from previously uploaded PDFs:
# from ehi_atlas.extract.uploads import list_uploads, get_upload
# for u in list_uploads():
#     print(f"  {u.hash_prefix}  {u.label!r}  ({u.size_bytes} bytes)")
# PDF_PATH = get_upload("<hash-prefix>").pdf_path
# ------------------------------------------------------------------------

# Cache + live-run controls (flip after review)
SKIP_CACHE = True            # True = always run live; False = use cache when present
PAGE_OF_INTEREST = 2         # which page to rasterize / inspect (1-indexed)

assert PDF_PATH.exists(), f"PDF not found at {PDF_PATH}"
print(f"PDF      : {PDF_PATH.relative_to(ATLAS_ROOT)}")
print(f"Size     : {PDF_PATH.stat().st_size:,} bytes")
print(f"Page     : {PAGE_OF_INTEREST}")
print(f"Skip cache: {SKIP_CACHE}")

# Optional: register this PDF in the uploads dropbox so it's findable later
# from ehi_atlas.extract.uploads import store_upload_from_path
# rec = store_upload_from_path(PDF_PATH, label="my run-2026-05-03")
# print(f"\nRegistered upload: {rec.hash_prefix}  {rec.label!r}")

## Step 1 — Rasterize a page

🔧 Script. Uses pdf2image (poppler) with a pure-Python `pypdfium2` fallback so this works on a fresh checkout without system deps.

In [ ]:
import pypdfium2 as pdfium

# In-memory rasterization at 150 DPI (readable in matplotlib, fast for live demos).
# layout.rasterize_pdf() writes to disk; we don't need that side effect here.
_doc = pdfium.PdfDocument(str(PDF_PATH))
_scale = 150 / 72.0  # PDF user units are 72 pt/inch
page_images = [page.render(scale=_scale).to_pil() for page in _doc]
print(f"Rendered {len(page_images)} page(s)")

img = page_images[PAGE_OF_INTEREST - 1]
print(f"Page {PAGE_OF_INTEREST} size: {img.size[0]} x {img.size[1]} px")

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 10))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"Page {PAGE_OF_INTEREST} — {PDF_PATH.name}", fontsize=11)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — image display skipped.")

## Step 2 — Extract text + bbox layer

🔧 Script. pdfplumber lifts every text span on every page with its bbox in PDF user units (bottom-left origin). The model sees the rasterized image; the bbox layer is the ground-truth grid the harmonizer can cross-check against.

In [ ]:
from ehi_atlas.extract.layout import extract_layout

doc_layout = extract_layout(PDF_PATH)
print(f"Pages: {len(doc_layout.pages)}")

page = doc_layout.pages[PAGE_OF_INTEREST - 1]
spans = page.spans
print(f"Page {PAGE_OF_INTEREST}: {len(spans)} spans, {page.width:.0f} x {page.height:.0f} pt\n")

for span in spans[:10]:
    print(f"  '{span.text}'  @  ({span.x1:.0f},{span.y1:.0f},{span.x2:.0f},{span.y2:.0f})")
if len(spans) > 10:
    print(f"  ... ({len(spans) - 10} more)")

## Step 3 — Find a known anchor

🔧 Script. `find_text_bbox` is the regression-test surface — given a known phrase, it returns the bbox so we can spot-check that the layout extractor still locates the row the gold extension is anchored to.

Default anchor: `"Creatinine"` (the rhett759 golden row at `page=2;bbox=72,574,540,590`). Swap `ANCHOR_TEXT` for any phrase you expect to find on `PAGE_OF_INTEREST`.

In [ ]:
from ehi_atlas.extract.layout import find_text_bbox

ANCHOR_TEXT = "Creatinine"

result = find_text_bbox(doc_layout, ANCHOR_TEXT, page=PAGE_OF_INTEREST)
if result is None:
    print(f"'{ANCHOR_TEXT}' not found on page {PAGE_OF_INTEREST}.")
    print("Try a different ANCHOR_TEXT (case-insensitive substring match).")
else:
    bbox = result.to_schemas_bbox()
    print(f"'{ANCHOR_TEXT}' found:")
    print(f"  locator: {bbox.to_locator_string()}")
    print(f"  x1={bbox.x1:.0f}, y1={bbox.y1:.0f}, x2={bbox.x2:.0f}, y2={bbox.y2:.0f}")
    if PDF_PATH.name == "lab-report-2025-09-12-quest.pdf":
        print("  expected (golden): page=2;bbox=72,574,540,590  (within ±5pt)")

## Step 4 — Cache check

Show the cache key for this PDF + the configured backend, and report whether an entry exists. The cache is content-hashed: same `(pdf_sha256, prompt_version, schema_version, backend/model)` tuple → same key → guaranteed deterministic replay.

In [ ]:
from ehi_atlas.extract.cache import ExtractionCache, CacheKey, hash_file
from ehi_atlas.extract.pdf import (
    DEFAULT_BACKEND, DEFAULT_MODEL,
    DEFAULT_PROMPT_VERSION, DEFAULT_SCHEMA_VERSION,
)

cache = ExtractionCache()
key = CacheKey(
    file_sha256=hash_file(PDF_PATH),
    prompt_version=DEFAULT_PROMPT_VERSION,
    schema_version=DEFAULT_SCHEMA_VERSION,
    model_name=f"{DEFAULT_BACKEND}/{DEFAULT_MODEL}",
)

print(f"backend/model    : {key.model_name}")
print(f"prompt version   : {key.prompt_version}")
print(f"schema version   : {key.schema_version}")
print(f"pdf sha256       : {key.file_sha256[:16]}...")
print(f"cache key digest : {key.digest()[:16]}...")
print(f"cache hit?       : {cache.has(key)}")

## Step 5 — Live extraction run 🤖

**This cell calls the configured backend.** Today that's `AnthropicBackend` (Claude); set the env var `EHI_VISION_BACKEND` to swap once a Gemma 4 backend lands.

By default this runs **fresh every time** (`SKIP_CACHE = True` at the top). The result is also written to the cache, so subsequent runs with `SKIP_CACHE = False` are free.

Required: `ANTHROPIC_API_KEY` in the environment (or in `ehi-atlas/.env`).

In [ ]:
import time
from ehi_atlas.extract.pdf import extract_lab_pdf

extraction_result = None
extraction_error = None

try:
    t0 = time.time()
    extraction_result = extract_lab_pdf(PDF_PATH, skip_cache=SKIP_CACHE)
    elapsed = time.time() - t0
    print(f"✓ Extraction succeeded in {elapsed:.2f}s")
    print(f"  document_type     : {extraction_result.document.document_type}")
    print(f"  extraction_model  : {extraction_result.extraction_model}")
    print(f"  prompt_version    : {extraction_result.extraction_prompt_version}")
    print(f"  confidence        : {extraction_result.extraction_confidence:.2f}")
    if extraction_result.document.document_type == "lab-report":
        report = extraction_result.document
        print(f"  lab               : {report.lab_name!r}")
        print(f"  document_date     : {report.document_date}")
        print(f"  results count     : {len(report.results)}")
        for i, r in enumerate(report.results[:5]):
            print(f"    [{i}] {r.test_name!r}  {r.value_quantity} {r.unit or ''}  flag={r.flag}  bbox={r.bbox.to_locator_string()}")
        if len(report.results) > 5:
            print(f"    ... ({len(report.results) - 5} more)")
except Exception as e:
    extraction_error = e
    print(f"✗ Extraction failed: {type(e).__name__}: {e}")
    print("\nCommon causes:")
    print("  - ANTHROPIC_API_KEY not set in environment / .env")
    print("  - Network blocked; backend cannot reach api.anthropic.com")
    print("  - Schema rejected by the model (see RuntimeError details above)")

In [ ]:
# Full validated ExtractionResult as JSON (for copying into a fixture or diffing)
if extraction_result is not None:
    print(extraction_result.model_dump_json(indent=2))

## Step 6 — Convert to FHIR Observation

🔧 Script — deterministic FHIR conversion, no LLM involved. Each `ExtractedLabResult` becomes a FHIR `Observation` with five `meta.extension` entries: extraction-model, extraction-confidence, extraction-prompt-version, source-attachment, and source-locator (the bbox string).

In [ ]:
from ehi_atlas.extract.to_fhir import lab_result_to_observation

if extraction_result is None or extraction_result.document.document_type != "lab-report":
    print("No lab-report extraction available — skipping FHIR conversion.")
else:
    report = extraction_result.document
    if not report.results:
        print("No lab results in the extraction — nothing to convert.")
    else:
        first = report.results[0]
        obs = lab_result_to_observation(
            result=first,
            patient_id="rhett759",
            source_attachment_id=PDF_PATH.stem,
            model=extraction_result.extraction_model,
            prompt_version=extraction_result.extraction_prompt_version,
            confidence=extraction_result.extraction_confidence,
        )
        print(f"resourceType  : {obs['resourceType']}")
        print(f"code          : {obs.get('code')}")
        print(f"valueQuantity : {obs.get('valueQuantity')}")
        print(f"effective     : {obs.get('effectiveDateTime')}")
        print()
        meta_exts = obs.get("meta", {}).get("extension", [])
        ext_urls = [e["url"].split("/")[-1] for e in meta_exts]
        print(f"meta.extension count: {len(meta_exts)}")
        print(f"Extension types     : {ext_urls}")
        locator = next(
            (e.get("valueString") for e in meta_exts if "source-locator" in e.get("url", "")),
            None,
        )
        print(f"source-locator      : {locator}")

**Next:** [04_layer3_code_mapping.ipynb](./04_layer3_code_mapping.ipynb)

**Want a UI?** The Streamlit PDF Lab at `app/pages/02b_PDF_Lab.py` does the same thing with a file uploader, side-by-side panels, and a bbox overlay. Run with:

```bash
uv run streamlit run app/streamlit_app.py --server.port 8503
```

Then click into **PDF Lab** in the sidebar.